# Mini scaling law for safety probes

Linear probes for jailbreak-vs-benign detection on small open models: learning curves (error vs. training budget D), a size trend, and robustness checks.

**v3 additions**
1. **Out-of-distribution test sets** (different source than the training data, filtered for exact/template/near-duplicate overlap). Learning curves, full-budget comparison and size trend are all reported on them.
2. **Budgets 4 and 8** added so the Hill half-saturation point D0 can actually be located.
3. **TF-IDF baseline on the same truncated text** the models see (plus the full-text one), for fair comparison, including in the evasion test.
4. **Split robustness**: the grouped split is repeated over several seeds; full-budget and D=64 errors, probe-vs-TF-IDF gaps and size exponents are reported with the across-split spread.
5. Hill diagnostics now use **test-set noise** (not just probe-seed variance); evasion CIs include a **Bonferroni-corrected** interval.

**Section 11 (appended follow-ups)**: power-law fits, deepset language check, attack-vs-benign shift split, stronger baselines, mean-pooled probes.

All v2 fixes are kept (dedupe + template-grouped split, token-level truncation, perturbations appended after truncation, standardized features with CV-tuned C, seeded runs, disk-cached features).

**Sections 12-13** (appended): mean-pooled readout as the main analysis with CIs, benign-coverage test, and ID-selected layer sweep. See the README for how to run this end to end.

In [ ]:
!pip install -q transformers datasets scikit-learn scipy matplotlib pandas

In [ ]:
import os, re, json, hashlib, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

SEED = 0                 # primary split / subsample seed
SPLIT_SEEDS = [0, 1, 2, 3, 4]   # for the split-robustness section
MAXLEN = 128             # tokens
N_SEEDS = 10             # subsample seeds per budget
N_BOOT = 2000            # bootstrap resamples over the test set
N_OOD_PER_CLASS = 400    # max OOD prompts per class
BASE_BUDGETS = [4, 8, 16, 32, 64, 128, 256, 512]   # final budget is always "all train"

MODELS = {
    "pythia-70m":   "EleutherAI/pythia-70m",
    "pythia-160m":  "EleutherAI/pythia-160m",
    "pythia-410m":  "EleutherAI/pythia-410m",
    "pythia-1b":    "EleutherAI/pythia-1b",
    "gpt2-small":   "gpt2",
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B",
}
PYTHIA = [m for m in MODELS if m.startswith("pythia")]
EVASION_MODELS = ["pythia-160m", "pythia-410m", "qwen2.5-0.5b"]
# name -> (suffix appended AFTER truncation, text transform applied BEFORE truncation)
PERTURBATIONS = {
    "please":            (" Please.", None),
    "filler":            (" Thanks! By the way, what is the weather like today?", None),
    "lowercase_nopunct": (None, lambda s: re.sub(r"[^\w\s]", "", s.lower())),
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LAYERS = ["mid", "last"]
print("device:", DEVICE)

In [ ]:
# Persist cached features and checkpoints on Google Drive so a Colab timeout does not lose them.
# If Drive cannot be mounted, falls back to /content (lost on timeout).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/safety_probe_cache"
except Exception as e:
    print("Drive not mounted, using /content:", type(e).__name__)
    PERSIST_DIR = "/content"
os.makedirs(PERSIST_DIR, exist_ok=True)
print("persistent dir:", PERSIST_DIR)

## 1. In-distribution data: dedupe, then split by template group

In [ ]:
from datasets import load_dataset
ds = load_dataset("jackhhao/jailbreak-classification", split="train")
df = pd.DataFrame({"prompt": ds["prompt"], "y": [int(t == "jailbreak") for t in ds["type"]]})
norm = lambda s: re.sub(r"\s+", " ", s.lower()).strip()
df["norm"] = df.prompt.map(norm)
n0 = len(df)
df = df.drop_duplicates("norm").reset_index(drop=True)
print(f"rows: {n0} -> {len(df)} after exact dedupe; jailbreak fraction = {df.y.mean():.2f}")

df["group"] = df.norm.map(lambda s: " ".join(s.split()[:8]))   # template family heuristic
P = df.prompt.values

def make_split(seed):
    return next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(df, df.y, groups=df.group))

tr, te = make_split(SEED)
ytr, yte = df.y.values[tr], df.y.values[te]
print(f"train={len(tr)}  test={len(te)}  test jailbreak frac={yte.mean():.2f}")

def make_bow():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                         LogisticRegression(C=10, max_iter=2000))

r_tr, r_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=df.y)
auc_rand = roc_auc_score(df.y.values[r_te], make_bow().fit(P[r_tr], df.y.values[r_tr]).decision_function(P[r_te]))
auc_grp  = roc_auc_score(yte, make_bow().fit(P[tr], ytr).decision_function(P[te]))
print(f"BoW AUROC  random split: {auc_rand:.3f} | grouped split: {auc_grp:.3f}")

## 2. Out-of-distribution sets

Candidates (each is skipped with a message if it cannot be loaded, e.g. no network or a dataset id changed):
- `itw_jailbreak`: TrustAIRLab in-the-wild jailbreak prompts (label 1) vs. regular prompts (label 0) from the same collection
- `deepset_injection`: deepset prompt-injection set (label 1 = injection). This is a **task shift** (injection, not only jailbreak), so read it as a stress test.

Hygiene: OOD prompts are dropped if they exact-match a training prompt, share a first-8-word template group with one, or have TF-IDF cosine >= 0.8 to any training prompt. Counts are printed so you can see how much overlap there was.

In [ ]:
ref_tfidf = TfidfVectorizer(sublinear_tf=True).fit(df.norm)
Xtrain_ref = ref_tfidf.transform(df.norm)
train_groups, train_norms = set(df.group), set(df.norm)

def load_ood_candidates():
    found = {}
    def itw():
        j = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", "jailbreak_2023_12_25", split="train")
        r = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", "regular_2023_12_25", split="train")
        return pd.DataFrame({"prompt": list(j["prompt"]) + list(r["prompt"]),
                             "y": [1] * len(j) + [0] * len(r)})
    def dsp():
        d = load_dataset("deepset/prompt-injections")
        parts = [d[s] for s in d]
        return pd.DataFrame({"prompt": sum([list(p["text"]) for p in parts], []),
                             "y": sum([list(p["label"]) for p in parts], [])})
    for key, fn in [("itw_jailbreak", itw), ("deepset_injection", dsp)]:
        try:
            found[key] = fn(); print(f"loaded '{key}': {len(found[key])} rows")
        except Exception as e:
            print(f"skip '{key}': {type(e).__name__}: {str(e)[:150]}")
    return found

def filter_ood(o, key, seed=SEED):
    o = o.dropna().copy(); o["prompt"] = o.prompt.astype(str); o["y"] = o.y.astype(int)
    o["norm"] = o.prompt.map(norm); o = o[o.norm.str.len() > 0].drop_duplicates("norm")
    n_start = len(o)
    o = o[~o.norm.isin(train_norms)]; n_exact = len(o)
    o["group"] = o.norm.map(lambda s: " ".join(s.split()[:8])); o = o[~o.group.isin(train_groups)]; n_grp = len(o)
    if len(o):
        sim = (ref_tfidf.transform(o.norm) @ Xtrain_ref.T).max(axis=1).toarray().ravel()
        o = o[sim < 0.8]
    print(f"  {key}: {n_start} -> exact {n_exact} -> template {n_grp} -> near-dup {len(o)}")
    rng = np.random.default_rng(seed); parts = []
    for c in (0, 1):
        oc = o[o.y == c]
        parts.append(oc.iloc[rng.permutation(len(oc))[:N_OOD_PER_CLASS]])
    o = pd.concat(parts).reset_index(drop=True)
    return o if min((o.y == 0).sum(), (o.y == 1).sum()) >= 20 else None

OOD = {}
for key, raw in load_ood_candidates().items():
    o = filter_ood(raw, key)
    if o is None: print(f"  dropping '{key}': too few examples per class after filtering")
    else: OOD[key] = o
OOD_COND = {k: f"ood_{k}_" + hashlib.md5("\n".join(o.prompt).encode()).hexdigest()[:8] for k, o in OOD.items()}
for k, o in OOD.items(): print(f"OOD '{k}': n={len(o)}, jailbreak/attack frac={o.y.mean():.2f}")
if not OOD: print("No OOD set available; OOD sections will be skipped. Add your own to OOD as a DataFrame(prompt, y).")

## 3. Activations (HF `transformers`, token-level truncation, cached to disk)

`mid` = hidden state after block n/2, `last` = final hidden state (post final norm). Features are taken at the last real token.

In [ ]:
from transformers import AutoTokenizer, AutoModel
CACHE = f"{PERSIST_DIR}/probe_cache_seed{SEED}_n{len(df)}"
os.makedirs(CACHE, exist_ok=True)
fpath = lambda name, cond: f"{CACHE}/{name}__{cond}.npz"
mpath = lambda name: f"{CACHE}/{name}__meta.json"

def encode(tok, prompts, suffix=None, transform=None):
    """Token-level truncation; suffix tokens are appended AFTER truncation so they are never cut."""
    suf = tok(suffix, add_special_tokens=False)["input_ids"] if suffix else []
    out = []
    for p in prompts:
        if transform: p = transform(p)
        ids = tok(p, add_special_tokens=False, truncation=True, max_length=MAXLEN - len(suf))["input_ids"] + suf
        out.append(ids if ids else [tok.eos_token_id])
    return out

@torch.no_grad()
def run_model(model, tok, ids_list, layer_idx, bs=16):
    feats = {k: [] for k in layer_idx}
    for i in range(0, len(ids_list), bs):
        batch = ids_list[i:i + bs]; L = max(map(len, batch))
        inp = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for j, ids in enumerate(batch):
            inp[j, :len(ids)] = torch.tensor(ids); att[j, :len(ids)] = 1
        out = model(input_ids=inp.to(DEVICE), attention_mask=att.to(DEVICE), output_hidden_states=True)
        last = (att.sum(1) - 1).to(DEVICE); ar = torch.arange(len(batch), device=DEVICE)
        for k, li in layer_idx.items():
            feats[k].append(out.hidden_states[li][ar, last].float().cpu())
    return {k: torch.cat(v).numpy() for k, v in feats.items()}

for name, hf in MODELS.items():
    todo = [("clean", None, None, P)]
    if name in EVASION_MODELS:
        todo += [(c, s, t, P[te]) for c, (s, t) in PERTURBATIONS.items()]
    todo += [(OOD_COND[k], None, None, o.prompt.values) for k, o in OOD.items()]
    todo = [x for x in todo if not os.path.exists(fpath(name, x[0]))]
    if not todo and os.path.exists(mpath(name)): continue
    print(f"[{name}] extracting: {[x[0] for x in todo]}")
    tok = AutoTokenizer.from_pretrained(hf); tok.padding_side = "right"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModel.from_pretrained(hf, torch_dtype=torch.float32).to(DEVICE).eval()
    n = model.config.num_hidden_layers; layer_idx = {"mid": n // 2, "last": n}
    n_emb = model.get_input_embeddings().weight.numel()
    json.dump({"n_params": model.num_parameters(), "n_nonemb": model.num_parameters() - n_emb,
               "d_model": model.config.hidden_size, "layer_idx": layer_idx}, open(mpath(name), "w"))
    for cond, suf, tf, texts in todo:
        np.savez(fpath(name, cond), **run_model(model, tok, encode(tok, texts, suf, tf), layer_idx))
    del model; torch.cuda.empty_cache()

META = {n: json.load(open(mpath(n))) for n in MODELS}
F = {n: {"clean": dict(np.load(fpath(n, "clean")))} for n in MODELS}
for n in MODELS:
    for k in OOD: F[n][f"ood:{k}"] = dict(np.load(fpath(n, OOD_COND[k])))
for n in EVASION_MODELS:
    for c in PERTURBATIONS: F[n][c] = dict(np.load(fpath(n, c)))
print(pd.DataFrame(META).T[["d_model", "n_params", "n_nonemb"]])

## 4. Utilities (no model dependencies)

In [ ]:
def fit_probe(X, y, seed):
    sc = StandardScaler().fit(X); Xs = sc.transform(X)
    if np.bincount(y, minlength=2).min() >= 3:
        clf = LogisticRegressionCV(Cs=[1e-3, 1e-2, 1e-1, 1.0], scoring="roc_auc", max_iter=2000,
                                   cv=StratifiedKFold(3, shuffle=True, random_state=seed), n_jobs=-1)
    else:
        clf = LogisticRegression(C=1e-2, max_iter=2000)
    clf.fit(Xs, y)
    return lambda Z: clf.decision_function(sc.transform(Z))

def fit_bow(texts, y):
    m = make_bow().fit(list(texts), y)
    return lambda T: m.decision_function(list(T))

def subsample(y, D, seed):
    if D >= len(y): return np.arange(len(y))
    rng = np.random.default_rng(seed); idx = []
    for c in (0, 1):
        ci = np.where(y == c)[0]
        idx.append(rng.choice(ci, size=min(len(ci), max(2, int(round(D * len(ci) / len(y))))), replace=False))
    return np.concatenate(idx)

def curve_multi(fit_fn, Xtr, ytr, evals, budgets, seeds):
    """evals: {name: (X, y)}. Returns {name: error matrix [n_seeds, n_budgets]}, error = 1 - AUROC."""
    E = {k: np.zeros((len(seeds), len(budgets))) for k in evals}
    for i, s in enumerate(seeds):
        for j, D in enumerate(budgets):
            idx = subsample(ytr, D, s); f = fit_fn(Xtr[idx], ytr[idx], s)
            for k, (Xe, ye) in evals.items():
                E[k][i, j] = 1 - roc_auc_score(ye, f(Xe))
    return E

def ci95(E):
    return E.mean(0), 1.96 * E.std(0, ddof=1) / np.sqrt(len(E))

def boot_auc(y, s, BI):
    out = np.full(len(BI), np.nan)
    for b, ix in enumerate(BI):
        if 0 < y[ix].sum() < len(ix): out[b] = roc_auc_score(y[ix], s[ix])
    return out

def hill(D, Ef, D0): return Ef + (0.5 - Ef) * D0 / (D + D0)

def fit_hill(D, E):
    try:
        p, _ = curve_fit(hill, D, E, p0=[max(min(E) * 0.8, 1e-3), 20.0],
                         bounds=([0, 0.5], [0.5, 1e5]), maxfev=20000)
        return p
    except Exception:
        return np.array([np.nan, np.nan])

def hill_report(budgets, E, test_noise, n_boot=300, seed=0):
    """test_noise = half-width of the 95% test-set bootstrap CI of the full-budget error."""
    rng = np.random.default_rng(seed)
    D = np.array(budgets, float); m = E.mean(0); p = fit_hill(D, m)
    bs = np.array([fit_hill(D, E[rng.integers(0, len(E), len(E))].mean(0)) for _ in range(n_boot)])
    lo, hi = np.nanpercentile(bs, [2.5, 97.5], axis=0)
    return dict(E_floor=p[0], E_floor_lo=lo[0], E_floor_hi=hi[0], D0=p[1], D0_lo=lo[1], D0_hi=hi[1],
                D0_in_range=bool(D.min() <= p[1] <= D.max()),
                test_noise=test_noise, floor_resolvable=bool(p[0] > test_noise),
                still_dropping=bool((m[-2] - m[-1]) > test_noise))

def compare_table(y, probe_scores, bow_scores, BIx):
    """probe_scores: {(model, layer): scores}; bow_scores: {label: scores}. Paired bootstrap on BIx."""
    bow_b = {k: boot_auc(y, s, BIx) for k, s in bow_scores.items()}
    for k, b in bow_b.items():
        print(f"  {k}: AUROC={roc_auc_score(y, bow_scores[k]):.3f}  CI [{np.nanpercentile(b,2.5):.3f}, {np.nanpercentile(b,97.5):.3f}]")
    rows = []
    for (m, l), s in probe_scores.items():
        b = boot_auc(y, s, BIx)
        r = dict(model=m, layer=l, auroc=roc_auc_score(y, s), ci_lo=np.nanpercentile(b, 2.5), ci_hi=np.nanpercentile(b, 97.5))
        for k, bb in bow_b.items():
            d = b - bb
            r[f"d_{k}"] = np.nanmean(d); r[f"d_{k}_lo"] = np.nanpercentile(d, 2.5); r[f"d_{k}_hi"] = np.nanpercentile(d, 97.5)
        rows.append(r)
    return pd.DataFrame(rows).set_index(["model", "layer"])

Reference tokenizer for the **truncated-text** TF-IDF control (the models only see the first 128 tokens, so the fair lexical baseline should see the same text).

In [ ]:
ref_tok = AutoTokenizer.from_pretrained("gpt2")
def trunc_text(p, suf=None, tf=None):
    if tf: p = tf(p)
    sid = ref_tok(suf, add_special_tokens=False)["input_ids"] if suf else []
    ids = ref_tok(p, add_special_tokens=False, truncation=True, max_length=MAXLEN - len(sid))["input_ids"]
    return ref_tok.decode(ids) + (suf or "")
T = lambda arr: np.array([trunc_text(p) for p in arr])
P_trunc = T(P)
OOD_text = {k: o.prompt.values for k, o in OOD.items()}
OOD_trunc = {k: T(v) for k, v in OOD_text.items()}
OOD_y = {k: o.y.values for k, o in OOD.items()}

## 5. Learning curves (in-distribution + OOD; probes vs. both TF-IDF baselines; same budgets and seeds)

In [ ]:
budgets = [b for b in BASE_BUDGETS if b < len(tr)] + [len(tr)]
seeds = list(range(N_SEEDS))
print("budgets:", budgets)

BI = np.random.default_rng(SEED).integers(0, len(te), size=(N_BOOT, len(te)))   # test-set bootstrap indices
BI_ood = {k: np.random.default_rng(SEED).integers(0, len(o), size=(N_BOOT, len(o))) for k, o in OOD.items()}

import pickle
CK = f"{CACHE}/curves_ckpt.pkl"
if os.path.exists(CK):
    bow_curves, bow_scores, curves, full_scores = pickle.load(open(CK, "rb"))
    print("loaded learning-curve checkpoint from", CK, "(delete the file to recompute)")
else:
    # TF-IDF baselines (full text and truncated text)
    bow_curves, bow_scores = {}, {}
    for label, Ptr_, Pid_, Pood_ in [("tfidf_full", P, P, OOD_text), ("tfidf_trunc", P_trunc, P_trunc, OOD_trunc)]:
        evals = {"id": (Pid_[te], yte), **{f"ood:{k}": (Pood_[k], OOD_y[k]) for k in OOD}}
        bow_curves[label] = curve_multi(lambda X, y, s: fit_bow(X, y), Ptr_[tr], ytr, evals, budgets, seeds)
        f = fit_bow(Ptr_[tr], ytr)
        bow_scores[label] = {"id": f(Pid_[te]), **{f"ood:{k}": f(Pood_[k]) for k in OOD}}

    curves, full_scores = {}, {}      # curves[(model, layer)][eval_key] -> [seeds, budgets]
    for name in MODELS:
        for layer in LAYERS:
            Xtr_ = F[name]["clean"][layer][tr]
            evals = {"id": (F[name]["clean"][layer][te], yte),
                     **{f"ood:{k}": (F[name][f"ood:{k}"][layer], OOD_y[k]) for k in OOD}}
            curves[(name, layer)] = curve_multi(fit_probe, Xtr_, ytr, evals, budgets, seeds)
            f = fit_probe(Xtr_, ytr, SEED)
            full_scores[(name, layer)] = {k: f(X) for k, (X, _) in evals.items()}
            msg = " ".join(f"{k}={curves[(name, layer)][k][:, -1].mean():.3f}" for k in evals)
            print(f"{name:13s} {layer:4s} err@all: {msg}")
    pickle.dump((bow_curves, bow_scores, curves, full_scores), open(CK, "wb"))
    print("saved checkpoint to", CK)


In [ ]:
eval_keys = ["id"] + [f"ood:{k}" for k in OOD]
fig, axes = plt.subplots(len(eval_keys), 2, figsize=(12, 4 * len(eval_keys)), sharey="row", squeeze=False)
for r, ek in enumerate(eval_keys):
    for c, layer in enumerate(LAYERS):
        ax = axes[r, c]
        for name in MODELS:
            m, h = ci95(curves[(name, layer)][ek])
            ax.plot(budgets, m, "o-", label=name, ms=3); ax.fill_between(budgets, m - h, m + h, alpha=0.15)
        for label, ls, col in [("tfidf_trunc", "k--", "k"), ("tfidf_full", "k:", "gray")]:
            m, h = ci95(bow_curves[label][ek]); ax.plot(budgets, m, ls, color=col, label=label)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(f"{ek} | layer: {layer}"); ax.set_xlabel("training budget D")
    axes[r, 0].set_ylabel("error = 1 - AUROC")
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("probe_scaling_v3.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Hill fits (in-distribution) with identifiability flags

- `D0_in_range`: half-saturation point lies inside the measured budgets (now that D=4 and 8 are included).
- `test_noise`: half-width of the test-set bootstrap CI of the full-budget error. `floor_resolvable=False` means the fitted floor is below this noise level, so it is not distinguishable from 0 on this test set.
- `still_dropping`: error fell between the last two budgets by more than `test_noise`.

In [ ]:
rows = []
for (name, layer), Ed in curves.items():
    b = 1 - boot_auc(yte, full_scores[(name, layer)]["id"], BI)
    noise = (np.nanpercentile(b, 97.5) - np.nanpercentile(b, 2.5)) / 2
    r = hill_report(budgets, Ed["id"], noise); r.update(model=name, layer=layer); rows.append(r)
hill_df = pd.DataFrame(rows).set_index(["model", "layer"])
print(hill_df.round(3).to_string())

## 7. Full-budget comparison vs. both TF-IDF baselines (paired bootstrap over the evaluation set)

A probe only beats lexical cues if the `d_tfidf_*` CI is entirely above 0.

In [ ]:
print("=== in-distribution (grouped test split) ===")
cmp_id = compare_table(yte, {k: v["id"] for k, v in full_scores.items()},
                       {l: s["id"] for l, s in bow_scores.items()}, BI)
print(cmp_id.round(3).to_string())

cmp_ood = {}
for k in OOD:
    print(f"\n=== OOD: {k} (n={len(OOD_y[k])}) ===")
    cmp_ood[k] = compare_table(OOD_y[k], {kk: v[f"ood:{k}"] for kk, v in full_scores.items()},
                               {l: s[f"ood:{k}"] for l, s in bow_scores.items()}, BI_ood[k])
    print(cmp_ood[k].round(3).to_string())

## 8. Size trend (Pythia only, measured full-budget error)

Four models, one family: treat the exponent as illustrative. Bootstrap resamples the same evaluation indices for all models.

In [ ]:
Ns = np.array([META[m]["n_nonemb"] for m in PYTHIA], float)

def size_trend(tag, y, get_scores, BIx, nb=1000):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax, layer in zip(axes, LAYERS):
        S = [get_scores(m, layer) for m in PYTHIA]
        E_pt = np.array([1 - roc_auc_score(y, s) for s in S]).clip(1e-4)
        slope = np.polyfit(np.log(Ns), np.log(E_pt), 1); al = []
        for ix in BIx[:nb]:
            if not (0 < y[ix].sum() < len(ix)): continue
            Eb = np.array([1 - roc_auc_score(y[ix], s[ix]) for s in S]).clip(1e-4)
            al.append(-np.polyfit(np.log(Ns), np.log(Eb), 1)[0])
        lo, hi = np.percentile(al, [2.5, 97.5])
        print(f"[{tag} | {layer}] alpha = {-slope[0]:.3f}  (95% boot CI [{lo:.3f}, {hi:.3f}]), errors = {np.round(E_pt, 3)}")
        ax.scatter(Ns, E_pt); g = np.logspace(np.log10(Ns.min()), np.log10(Ns.max()), 50)
        ax.plot(g, np.exp(np.polyval(slope, np.log(g))), "k--", label=f"alpha={-slope[0]:.2f}")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("non-embedding params")
        ax.set_ylabel("error @ full budget"); ax.set_title(f"{tag} | {layer}"); ax.legend()
    plt.tight_layout(); plt.savefig(f"size_law_{tag.replace(':','_')}.png", dpi=150, bbox_inches="tight"); plt.show()

size_trend("id", yte, lambda m, l: full_scores[(m, l)]["id"], BI)
for k in OOD:
    size_trend(f"ood:{k}", OOD_y[k], lambda m, l, k=k: full_scores[(m, l)][f"ood:{k}"], BI_ood[k])

## 9. Probe evasion (perturbations survive truncation; both TF-IDF controls; Bonferroni-corrected CIs)

Probes are trained on clean train data at full budget and tested on perturbed test prompts. `sig_bonf=True` means the Bonferroni-corrected CI of the AUROC drop excludes 0. These are *non-adversarial* perturbations.

In [ ]:
def perturb_text(p, suf, tf):
    p = tf(p) if tf else p
    return p + (suf or "")

rows, ds_ = [], []
def add(model, layer, pert, s_clean, s_pert):
    d = boot_auc(yte, s_clean, BI) - boot_auc(yte, s_pert, BI)
    rows.append(dict(model=model, layer=layer, pert=pert, clean=roc_auc_score(yte, s_clean),
                     perturbed=roc_auc_score(yte, s_pert), drop=np.nanmean(d),
                     lo=np.nanpercentile(d, 2.5), hi=np.nanpercentile(d, 97.5))); ds_.append(d)

for name in EVASION_MODELS:
    for layer in LAYERS:
        f = fit_probe(F[name]["clean"][layer][tr], ytr, SEED)
        s_clean = f(F[name]["clean"][layer][te])
        for cond in PERTURBATIONS: add(name, layer, cond, s_clean, f(F[name][cond][layer]))

bf_full, bf_trunc = fit_bow(P[tr], ytr), fit_bow(P_trunc[tr], ytr)
for cond, (suf, tf) in PERTURBATIONS.items():
    add("TF-IDF full", "-", cond, bf_full(P[te]), bf_full([perturb_text(p, suf, tf) for p in P[te]]))
    add("TF-IDF trunc", "-", cond, bf_trunc(P_trunc[te]), bf_trunc([trunc_text(p, suf, tf) for p in P[te]]))

a = 0.05 / len(rows)
for r, d in zip(rows, ds_):
    r["lo_bonf"] = np.nanpercentile(d, 100 * a / 2); r["hi_bonf"] = np.nanpercentile(d, 100 - 100 * a / 2)
    r["sig_bonf"] = bool(r["lo_bonf"] > 0)
ev_df = pd.DataFrame(rows)
print(f"{len(rows)} comparisons; Bonferroni alpha = {a:.4f}")
print(ev_df.round(3).to_string(index=False))

## 10. Split robustness

The grouped split is repeated with several seeds (features are cached, so only probes are refit). Shows whether conclusions depend on one particular test split. Errors are reported at the full budget and at D=64 (where the curves are less saturated).

In [ ]:
rows = []
for s in SPLIT_SEEDS:
    tr_s, te_s = make_split(s); ytr_s, yte_s = df.y.values[tr_s], df.y.values[te_s]
    e_bf = 1 - roc_auc_score(yte_s, fit_bow(P[tr_s], ytr_s)(P[te_s]))
    e_bt = 1 - roc_auc_score(yte_s, fit_bow(P_trunc[tr_s], ytr_s)(P_trunc[te_s]))
    for name in MODELS:
        for layer in LAYERS:
            X = F[name]["clean"][layer]; Xtr_, Xte_ = X[tr_s], X[te_s]
            e_all = 1 - roc_auc_score(yte_s, fit_probe(Xtr_, ytr_s, s)(Xte_))
            e64 = np.mean([1 - roc_auc_score(yte_s, fit_probe(Xtr_[ix], ytr_s[ix], sd)(Xte_))
                           for sd in range(3) for ix in [subsample(ytr_s, 64, sd)]])
            rows.append(dict(split=s, model=name, layer=layer, err_all=e_all, err_64=e64,
                             gap_vs_trunc=e_all - e_bt, gap_vs_full=e_all - e_bf, bow_full_err=e_bf, bow_trunc_err=e_bt))
sp = pd.DataFrame(rows)
summ = sp.groupby(["model", "layer"])[["err_all", "err_64", "gap_vs_trunc", "gap_vs_full"]].agg(["mean", "std"])
print("Across-split mean/std (gap > 0 means the probe is WORSE than that TF-IDF baseline):")
print(summ.round(3).to_string())
print("\nTF-IDF error per split:\n", sp.groupby("split")[["bow_full_err", "bow_trunc_err"]].first().round(3).to_string())

print("\nSize exponent alpha per split (Pythia, error ~ N^-alpha):")
arows = []
for s, g in sp.groupby("split"):
    for layer in LAYERS:
        gg = g[(g.layer == layer) & g.model.isin(PYTHIA)].set_index("model").loc[PYTHIA]
        for col in ["err_all", "err_64"]:
            arows.append(dict(split=s, layer=layer, at=col,
                              alpha=-np.polyfit(np.log(Ns), np.log(gg[col].values.clip(1e-4)), 1)[0]))
adf = pd.DataFrame(arows)
print(adf.pivot_table(index=["layer", "at"], columns="split", values="alpha").round(3).to_string())
print(adf.groupby(["layer", "at"]).alpha.agg(["mean", "std", "min", "max"]).round(3).to_string())

## How to read the results, and what still limits them

- **Probe vs. lexical cues:** use section 7. On in-distribution data TF-IDF was near ceiling, so the OOD tables are the real test. A probe adds something only if `d_tfidf_*` is positive with a CI above 0 there.
- **Scaling:** trust a Hill fit only if `D0_in_range` and `floor_resolvable`; trust the size exponent only if its CI (section 8) and its spread across splits (section 10) exclude 0 and agree. Four Pythia points can show at most an illustrative trend.
- **Evasion:** only non-adversarial perturbations; compare each probe with the truncated TF-IDF control, and rely on `sig_bonf`, not the plain CI.
- **Remaining limits:** small datasets (~1k in-distribution, <=800 OOD), template grouping is a first-8-words heuristic, the deepset OOD set is a task shift (injection, not jailbreak), OOD benign prompts come from a different distribution too (so AUROC reflects both attack and benign shift), base models only, last-token readout, and perturbation features are cached only for the primary split.

## 11. Follow-ups (appended to this notebook; reuses cached features, so most cells are cheap)

11a. Replace the Hill fit with an offset power law (the Hill form was misspecified: fitted floors exceeded measured errors) and tabulate measured errors.
11b. Language check on the deepset OOD set (is the "scaling" there just English-vs-German?).
11c. Split OOD shift into *attack shift* vs *benign shift*.
11d. Stronger lexical/semantic baselines (char n-gram TF-IDF, MiniLM embeddings).
11e. Mean-pooled probes (readout-position control for the evasion fragility). Set `RUN_MEAN_POOL = False` to skip.

### 11a. Offset power law `E = a * D^-b + c` instead of Hill, plus measured errors

In [ ]:
def powlaw(D, a, b, c): return a * np.power(D, -b) + c

def fit_pl(D, E):
    try:
        p, _ = curve_fit(powlaw, D, E, p0=[0.5, 0.5, max(min(E) * 0.5, 1e-4)],
                         bounds=([0, 0, 0], [10, 3, 0.5]), maxfev=20000)
        return p
    except Exception:
        return np.array([np.nan] * 3)

Dd = np.array(budgets, float); rows = []
for (name, layer), Ed in curves.items():
    for ek in eval_keys:
        E = Ed[ek]; m = E.mean(0); p = fit_pl(Dd, m); ph = fit_hill(Dd, m)
        rng = np.random.default_rng(0)
        bs = np.array([fit_pl(Dd, E[rng.integers(0, len(E), len(E))].mean(0))[1] for _ in range(200)])
        rows.append(dict(model=name, layer=layer, eval=ek, err_16=m[budgets.index(16)], err_64=m[budgets.index(64)],
                         err_all=m[-1], b=p[1], b_lo=np.nanpercentile(bs, 2.5), b_hi=np.nanpercentile(bs, 97.5), c=p[2],
                         rmse_powlaw=np.sqrt(np.mean((powlaw(Dd, *p) - m) ** 2)),
                         rmse_hill=np.sqrt(np.mean((hill(Dd, *ph) - m) ** 2))))
pl_df = pd.DataFrame(rows).set_index(["eval", "model", "layer"])
print("b = data-scaling exponent (error ~ D^-b); compare rmse_powlaw vs rmse_hill for fit quality")
print(pl_df.round(3).to_string())

### 11b. deepset OOD: language check (crude stopword heuristic)

In [ ]:
GER = set("der die das und ist nicht ein eine zu den von mit für auf ich sie wie was wer wird sind auch dem im es du wir war hat".split())
ENG = set("the and is not a to of in that it for you with on are as this be what how".split())
def lang(s):
    w = re.findall(r"[a-zäöüß]+", s.lower()); g = sum(x in GER for x in w); e = sum(x in ENG for x in w)
    return "de" if g > e else ("en" if e > 0 else "other")

def subset_compare(k, mask, tag):
    y = OOD_y[k][mask]; n_ = int(mask.sum())
    if min(y.sum(), n_ - y.sum()) < 15: print(f"{tag}: too few examples in one class ({n_} total)"); return
    BIs = np.random.default_rng(SEED).integers(0, n_, size=(N_BOOT, n_))
    print(f"--- {k} | {tag} (n={n_}, attack frac={y.mean():.2f}) ---")
    out = compare_table(y, {kk: v[f"ood:{k}"][mask] for kk, v in full_scores.items()},
                        {l: s[f"ood:{k}"][mask] for l, s in bow_scores.items()}, BIs)
    print(out[["auroc", "ci_lo", "ci_hi", "d_tfidf_trunc", "d_tfidf_trunc_lo", "d_tfidf_trunc_hi"]].round(3).to_string())

if "deepset_injection" in OOD:
    L = OOD["deepset_injection"].prompt.map(lang).values
    print(pd.Series(L).value_counts().to_dict(), "| attack frac by language:",
          {l: round(float(OOD_y["deepset_injection"][L == l].mean()), 2) for l in set(L)})
    subset_compare("deepset_injection", L == "en", "English only")
    subset_compare("deepset_injection", L == "de", "German only")
else:
    print("deepset_injection not loaded; skipping.")

### 11c. Attack shift vs. benign shift

Scores come from the same probe trained on in-distribution train data, so they are comparable across sets. `attack_shift` = OOD attacks vs. in-dist benign; `benign_shift` = in-dist attacks vs. OOD benign; `both` = full OOD.

In [ ]:
def auc_pn(pos, neg): return roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg])
def fmt(pos, neg, nb=300, seed=0):
    rng = np.random.default_rng(seed)
    b = [auc_pn(pos[rng.integers(0, len(pos), len(pos))], neg[rng.integers(0, len(neg), len(neg))]) for _ in range(nb)]
    lo, hi = np.percentile(b, [2.5, 97.5]); return f"{auc_pn(pos, neg):.3f} [{lo:.2f},{hi:.2f}]"

def decomp_rows(items, k):
    y_o = OOD_y[k]; rows = []
    for label, s_id, s_o in items:
        rows.append({"scorer": label,
                     "in_dist": fmt(s_id[yte == 1], s_id[yte == 0]),
                     "attack_shift": fmt(s_o[y_o == 1], s_id[yte == 0]),
                     "benign_shift": fmt(s_id[yte == 1], s_o[y_o == 0]),
                     "both": fmt(s_o[y_o == 1], s_o[y_o == 0])})
    return pd.DataFrame(rows).set_index("scorer")

for k in OOD:
    items = [(f"{m} {l}", v["id"], v[f"ood:{k}"]) for (m, l), v in full_scores.items()]
    items += [(l, s["id"], s[f"ood:{k}"]) for l, s in bow_scores.items()]
    print(f"\n=== {k}: where does the OOD drop come from? ===")
    print(decomp_rows(items, k).to_string())

### 11d. Stronger baselines: char n-gram TF-IDF and MiniLM embeddings (on truncated text)

In [ ]:
import subprocess, sys
def fit_char(texts, y):
    m = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, max_features=200000),
                      LogisticRegression(C=10, max_iter=2000)).fit(list(texts), y)
    return lambda T: m.decision_function(list(T))

strong = {}
f = fit_char(P_trunc[tr], ytr)
strong["char_tfidf_trunc"] = {"id": f(P_trunc[te]), **{f"ood:{k}": f(OOD_trunc[k]) for k in OOD}}
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    emb = lambda arr: st.encode(list(arr), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    Etr, Ete = emb(P_trunc[tr]), emb(P_trunc[te])
    f = fit_probe(Etr, ytr, SEED)
    strong["minilm_probe"] = {"id": f(Ete), **{f"ood:{k}": f(emb(OOD_trunc[k])) for k in OOD}}
except Exception as e:
    print("MiniLM baseline skipped:", type(e).__name__, str(e)[:150])

best_probes = [("pythia-1b", "last"), ("pythia-1b", "mid"), ("qwen2.5-0.5b", "last"), ("qwen2.5-0.5b", "mid")]
ref_bow = {**{l: s for l, s in bow_scores.items()}, **strong}
for ek, y_, BIx in [("id", yte, BI)] + [(f"ood:{k}", OOD_y[k], BI_ood[k]) for k in OOD]:
    print(f"\n=== {ek}: best probes vs. stronger baselines (d_* > 0 with CI above 0 means probe wins) ===")
    out = compare_table(y_, {bp: full_scores[bp][ek] for bp in best_probes}, {l: s[ek] for l, s in ref_bow.items()}, BIx)
    cols = ["auroc"] + [c for c in out.columns if c.startswith("d_") and not c.endswith(("_lo", "_hi"))]
    print(out[cols].round(3).to_string())

### 11e. Mean-pooled probes (readout-position control)

If mean-pooled features are much more robust to the appended-text perturbations than last-token features, the earlier fragility is a readout-position effect rather than a probe-fooling effect.

In [ ]:
RUN_MEAN_POOL = True

@torch.no_grad()
def run_model_mean(model, tok, ids_list, layer_idx, bs=16):
    feats = {k: [] for k in layer_idx}
    for i in range(0, len(ids_list), bs):
        batch = ids_list[i:i + bs]; L = max(map(len, batch))
        inp = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for j, ids in enumerate(batch):
            inp[j, :len(ids)] = torch.tensor(ids); att[j, :len(ids)] = 1
        out = model(input_ids=inp.to(DEVICE), attention_mask=att.to(DEVICE), output_hidden_states=True)
        a = att.to(DEVICE).unsqueeze(-1).float()
        for k, li in layer_idx.items():
            h = out.hidden_states[li].float()
            feats[k].append(((h * a).sum(1) / a.sum(1)).cpu())
    return {k: torch.cat(v).numpy() for k, v in feats.items()}

if RUN_MEAN_POOL:
    FM = {}
    for name, hf in MODELS.items():
        todo = [("mean_clean", None, None, P)]
        if name in EVASION_MODELS: todo += [(f"mean_{c}", s, t, P[te]) for c, (s, t) in PERTURBATIONS.items()]
        todo += [(f"mean_{OOD_COND[k]}", None, None, o.prompt.values) for k, o in OOD.items()]
        todo = [x for x in todo if not os.path.exists(fpath(name, x[0]))]
        if todo:
            print(f"[{name}] mean-pool extracting: {[x[0] for x in todo]}")
            tok = AutoTokenizer.from_pretrained(hf); tok.padding_side = "right"
            if tok.pad_token is None: tok.pad_token = tok.eos_token
            model = AutoModel.from_pretrained(hf, torch_dtype=torch.float32).to(DEVICE).eval()
            for cond, suf, tf, texts in todo:
                np.savez(fpath(name, cond), **run_model_mean(model, tok, encode(tok, texts, suf, tf), META[name]["layer_idx"]))
            del model; torch.cuda.empty_cache()
        FM[name] = {"clean": dict(np.load(fpath(name, "mean_clean")))}
        for k in OOD: FM[name][f"ood:{k}"] = dict(np.load(fpath(name, f"mean_{OOD_COND[k]}")))
        if name in EVASION_MODELS:
            for c in PERTURBATIONS: FM[name][c] = dict(np.load(fpath(name, f"mean_{c}")))

    rows = []
    for name in MODELS:
        for layer in LAYERS:
            f = fit_probe(FM[name]["clean"][layer][tr], ytr, SEED)
            r = dict(model=name, layer=layer, id_mean=roc_auc_score(yte, f(FM[name]["clean"][layer][te])),
                     id_last=roc_auc_score(yte, full_scores[(name, layer)]["id"]))
            for k in OOD:
                r[f"{k}_mean"] = roc_auc_score(OOD_y[k], f(FM[name][f"ood:{k}"][layer]))
                r[f"{k}_last"] = roc_auc_score(OOD_y[k], full_scores[(name, layer)][f"ood:{k}"])
            rows.append(r)
    print("Full-budget AUROC: mean-pooled vs last-token readout")
    print(pd.DataFrame(rows).set_index(["model", "layer"]).round(3).to_string())

    rows = []
    for name in EVASION_MODELS:
        for layer in LAYERS:
            f = fit_probe(FM[name]["clean"][layer][tr], ytr, SEED); s_c = f(FM[name]["clean"][layer][te])
            for cond in PERTURBATIONS:
                s_p = f(FM[name][cond][layer]); d = boot_auc(yte, s_c, BI) - boot_auc(yte, s_p, BI)
                s_l = full_scores[(name, layer)]["id"]
                f_l = fit_probe(F[name]["clean"][layer][tr], ytr, SEED)
                d_l = boot_auc(yte, s_l, BI) - boot_auc(yte, f_l(F[name][cond][layer]), BI)
                rows.append(dict(model=name, layer=layer, pert=cond, drop_mean=np.nanmean(d),
                                 mean_lo=np.nanpercentile(d, 2.5), mean_hi=np.nanpercentile(d, 97.5),
                                 drop_last=np.nanmean(d_l)))
    print("\nEvasion: AUROC drop with mean-pooled vs last-token features (plain 95% CIs, no multiple-comparison correction)")
    print(pd.DataFrame(rows).round(3).to_string(index=False))
else:
    print("RUN_MEAN_POOL = False; skipping.")

## How to read the follow-ups

- **11a:** report measured errors and `b`; ignore Hill's `E_floor`/`D0` unless `rmse_hill` is as small as `rmse_powlaw`.
- **11b:** if the deepset advantage disappears on the English subset, it was largely a language effect for TF-IDF, not attack detection.
- **11c:** tells you whether the OOD collapse comes from new attack styles or from different benign prompts.
- **11d:** "beats TF-IDF" is only meaningful if the probe also beats the char n-gram and embedding baselines.
- **11e:** if mean-pooled probes drop much less under the perturbations, the fragility was a last-token readout artifact.

## 12. Mean-pooled readout as the main analysis (appended)

Section 11e showed that mean-pooled features are robust to the appended-text perturbations and match TF-IDF in-distribution. This section redoes the scaling analyses on that readout and adds confidence intervals to the strong-baseline comparison.

**Prerequisites (run these first):** Sections 1-5 (checkpoint makes 5 instant), 11d (defines `strong`), 11e (creates the cached mean-pooled features). Then run 12a to 12d in order.

- 12a: learning curves with mean-pooled features (checkpointed to Drive)
- 12b: probe vs. TF-IDF / char n-gram / MiniLM **with 95% CIs** (`*` = probe significantly better, `x` = significantly worse), for mean-pooled and last-token readouts
- 12c: size trend and power-law exponents for mean-pooled probes
- 12d: split robustness for mean-pooled probes

### 12a. Learning curves, mean-pooled features

In [ ]:
import pickle
FM = {}
for n in MODELS:
    assert os.path.exists(fpath(n, "mean_clean")), "Mean-pooled features missing: run Section 11e first."
    FM[n] = {"clean": dict(np.load(fpath(n, "mean_clean")))}
    for k in OOD: FM[n][f"ood:{k}"] = dict(np.load(fpath(n, f"mean_{OOD_COND[k]}")))

CKM = f"{CACHE}/curves_mean_ckpt.pkl"
if os.path.exists(CKM):
    curves_m, full_scores_m = pickle.load(open(CKM, "rb"))
    print("loaded mean-pooled checkpoint from", CKM, "(delete the file to recompute)")
else:
    curves_m, full_scores_m = {}, {}
    for name in MODELS:
        for layer in LAYERS:
            Xtr_ = FM[name]["clean"][layer][tr]
            evals = {"id": (FM[name]["clean"][layer][te], yte),
                     **{f"ood:{k}": (FM[name][f"ood:{k}"][layer], OOD_y[k]) for k in OOD}}
            curves_m[(name, layer)] = curve_multi(fit_probe, Xtr_, ytr, evals, budgets, seeds)
            f = fit_probe(Xtr_, ytr, SEED)
            full_scores_m[(name, layer)] = {k: f(X) for k, (X, _) in evals.items()}
            msg = " ".join(f"{k}={curves_m[(name, layer)][k][:, -1].mean():.3f}" for k in evals)
            print(f"{name:13s} {layer:4s} err@all (mean-pooled): {msg}")
    pickle.dump((curves_m, full_scores_m), open(CKM, "wb")); print("saved checkpoint to", CKM)

eval_keys = ["id"] + [f"ood:{k}" for k in OOD]
fig, axes = plt.subplots(len(eval_keys), 2, figsize=(12, 4 * len(eval_keys)), sharey="row", squeeze=False)
for r, ek in enumerate(eval_keys):
    for c, layer in enumerate(LAYERS):
        ax = axes[r, c]
        for name in MODELS:
            m, h = ci95(curves_m[(name, layer)][ek])
            ax.plot(budgets, m, "o-", label=name, ms=3); ax.fill_between(budgets, m - h, m + h, alpha=0.15)
        for label, ls, col in [("tfidf_trunc", "k--", "k"), ("tfidf_full", "k:", "gray")]:
            m, h = ci95(bow_curves[label][ek]); ax.plot(budgets, m, ls, color=col, label=label)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(f"MEAN-POOLED | {ek} | layer: {layer}"); ax.set_xlabel("training budget D")
    axes[r, 0].set_ylabel("error = 1 - AUROC")
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("probe_scaling_meanpool.png", dpi=150, bbox_inches="tight"); plt.show()

### 12b. Probe vs. baselines with 95% CIs (paired bootstrap)

`*` = CI entirely above 0 (probe better), `x` = CI entirely below 0 (probe worse), blank = not distinguishable.

In [ ]:
def fmtd(d, lo, hi, w=True):
    mark = " *" if lo > 0 else (" x" if hi < 0 else "")
    return f"{d:+.3f} [{lo:+.3f},{hi:+.3f}]{mark}"

base_all = {**bow_scores, **strong}
REF = [l for l in ("tfidf_trunc", "char_tfidf_trunc", "minilm_probe") if l in base_all]

def ci_table(readout_scores, ek, y, BIx):
    ref = {l: base_all[l][ek] for l in REF}
    out = compare_table(y, {ml: readout_scores[ml][ek] for ml in readout_scores}, ref, BIx)
    rows = []
    for idx, r in out.iterrows():
        row = {"auroc": f"{r.auroc:.3f}"}
        for l in REF: row[f"vs_{l}"] = fmtd(r[f"d_{l}"], r[f"d_{l}_lo"], r[f"d_{l}_hi"])
        rows.append(pd.Series(row, name=idx))
    return pd.DataFrame(rows)

for ek, y_, BIx in [("id", yte, BI)] + [(f"ood:{k}", OOD_y[k], BI_ood[k]) for k in OOD]:
    for tag, sc in [("MEAN-POOLED", full_scores_m), ("LAST-TOKEN", full_scores)]:
        print(f"\n=== {ek} | {tag} readout ===")
        print(ci_table(sc, ek, y_, BIx).to_string())

### 12c. Size trend and power-law exponents (mean-pooled)

In [ ]:
for tag, y_, gs, BIx in [("id_mean", yte, lambda m, l: full_scores_m[(m, l)]["id"], BI)] + \
        [(f"ood:{k}_mean", OOD_y[k], (lambda m, l, k=k: full_scores_m[(m, l)][f"ood:{k}"]), BI_ood[k]) for k in OOD]:
    size_trend(tag, y_, gs, BIx)

Dd = np.array(budgets, float); rows = []
for (name, layer), Ed in curves_m.items():
    for ek in eval_keys:
        E = Ed[ek]; m = E.mean(0); p = fit_pl(Dd, m); rng = np.random.default_rng(0)
        bs = np.array([fit_pl(Dd, E[rng.integers(0, len(E), len(E))].mean(0))[1] for _ in range(200)])
        rows.append(dict(eval=ek, model=name, layer=layer, err_16=m[budgets.index(16)], err_64=m[budgets.index(64)],
                         err_all=m[-1], b=p[1], b_lo=np.nanpercentile(bs, 2.5), b_hi=np.nanpercentile(bs, 97.5), c=p[2]))
print("\nMean-pooled power-law fits (error ~ a*D^-b + c):")
print(pd.DataFrame(rows).set_index(["eval", "model", "layer"]).round(3).to_string())

### 12d. Split robustness (mean-pooled)

In [ ]:
rows = []
for s in SPLIT_SEEDS:
    tr_s, te_s = make_split(s); ytr_s, yte_s = df.y.values[tr_s], df.y.values[te_s]
    e_bt = 1 - roc_auc_score(yte_s, fit_bow(P_trunc[tr_s], ytr_s)(P_trunc[te_s]))
    for name in MODELS:
        for layer in LAYERS:
            X = FM[name]["clean"][layer]; Xtr_, Xte_ = X[tr_s], X[te_s]
            e_all = 1 - roc_auc_score(yte_s, fit_probe(Xtr_, ytr_s, s)(Xte_))
            e64 = np.mean([1 - roc_auc_score(yte_s, fit_probe(Xtr_[ix], ytr_s[ix], sd)(Xte_))
                           for sd in range(3) for ix in [subsample(ytr_s, 64, sd)]])
            rows.append(dict(split=s, model=name, layer=layer, err_all=e_all, err_64=e64,
                             gap_vs_trunc=e_all - e_bt, bow_trunc_err=e_bt))
sp_m = pd.DataFrame(rows)
print("Mean-pooled, across-split mean/std (gap > 0: probe WORSE than truncated TF-IDF):")
print(sp_m.groupby(["model", "layer"])[["err_all", "err_64", "gap_vs_trunc"]].agg(["mean", "std"]).round(3).to_string())
arows = []
for s, g in sp_m.groupby("split"):
    for layer in LAYERS:
        gg = g[(g.layer == layer) & g.model.isin(PYTHIA)].set_index("model").loc[PYTHIA]
        for col in ["err_all", "err_64"]:
            arows.append(dict(split=s, layer=layer, at=col,
                              alpha=-np.polyfit(np.log(Ns), np.log(gg[col].values.clip(1e-4)), 1)[0]))
adf = pd.DataFrame(arows)
print("\nSize exponent alpha per split (mean-pooled, Pythia):")
print(adf.pivot_table(index=["layer", "at"], columns="split", values="alpha").round(3).to_string())
print(adf.groupby(["layer", "at"]).alpha.agg(["mean", "std", "min", "max"]).round(3).to_string())

## How to read Section 12

- **12b:** the key question is whether any probe row has `*` against `char_tfidf_trunc` and `minilm_probe` on the OOD sets. `x` against MiniLM on itw means the probe is significantly worse than a small sentence-embedding model there.
- **12c / 12d:** trust the size exponent only if its CI excludes 0 and the per-split alphas agree. If mean-pooled ID error is near 0 for all models (saturated), the ID size trend is not identifiable; the OOD deepset trend is the informative one.
- Still limited by: 4 Pythia models, ~1k training prompts, one in-distribution source, heuristic template grouping, base models only, and the deepset task shift.

## 13. Final checks (appended)

- **13a** benign-coverage test: does adding some in-the-wild benign prompts to training remove the OOD error floor on the in-the-wild set (held-out, group-disjoint)?
- **13b** extract mean-pooled features for all layers (cached once).
- **13c** layer sweep: the layer is selected using in-distribution grouped CV only, then OOD is reported for that layer (avoids choosing the layer by looking at OOD results).

In [ ]:
# 13a. Benign-coverage test: does adding itw benign prompts to training fix the itw OOD floor?
from sklearn.model_selection import GroupShuffleSplit

KEY = "itw_jailbreak"; assert KEY in OOD, "itw_jailbreak OOD set not loaded"
o, yo = OOD[KEY], OOD_y[KEY]
o_tr, o_te = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED).split(o, yo, groups=o.group.values))
print(f"itw pool: train {len(o_tr)} (attack frac {yo[o_tr].mean():.2f}) | held-out {len(o_te)} (attack frac {yo[o_te].mean():.2f})")
assert 0 < yo[o_te].mean() < 1

K_LIST = [0, 25, 50, 100, 200]
MODES = ["benign_only", "benign_plus_attacks"]   # add K itw benign  |  add K itw benign + K itw attacks
SEEDS_C = [0, 1, 2]
pool_b, pool_a = o_tr[yo[o_tr] == 0], o_tr[yo[o_tr] == 1]
EMPTY = np.array([], dtype=int)

def pick(K, mode, seed):
    rng = np.random.default_rng(seed)
    b = rng.choice(pool_b, size=min(K, len(pool_b)), replace=False) if K > 0 else EMPTY
    a = rng.choice(pool_a, size=min(K, len(pool_a)), replace=False) if (K > 0 and mode != "benign_only") else EMPTY
    return np.concatenate([b, a]).astype(int)

DS = "deepset_injection" if "deepset_injection" in OOD else None
rows = []
for name in MODELS:
    for layer in LAYERS:
        Xc, Xo = FM[name]["clean"][layer], FM[name][f"ood:{KEY}"][layer]
        Xd = FM[name][f"ood:{DS}"][layer] if DS else None
        for mode in MODES:
            for K in K_LIST:
                if K == 0 and mode != "benign_only": continue      # K=0 is identical for both modes
                for s in SEEDS_C:
                    ex = pick(K, mode, s)
                    Xtr_ = np.vstack([Xc[tr], Xo[ex]]); ytr_ = np.r_[ytr, yo[ex]].astype(int)
                    f = fit_probe(Xtr_, ytr_, s)
                    r = dict(model=name, layer=layer, mode=mode, K=K, seed=s,
                             itw_heldout=1 - roc_auc_score(yo[o_te], f(Xo[o_te])),
                             id=1 - roc_auc_score(yte, f(Xc[te])))
                    if DS: r["deepset"] = 1 - roc_auc_score(OOD_y[DS], f(Xd))
                    rows.append(r)

# TF-IDF (truncated text) under the same protocol
for mode in MODES:
    for K in K_LIST:
        if K == 0 and mode != "benign_only": continue
        for s in SEEDS_C:
            ex = pick(K, mode, s)
            f = fit_bow(np.r_[P_trunc[tr], OOD_trunc[KEY][ex]], np.r_[ytr, yo[ex]].astype(int))
            r = dict(model="tfidf_trunc", layer="-", mode=mode, K=K, seed=s,
                     itw_heldout=1 - roc_auc_score(yo[o_te], f(OOD_trunc[KEY][o_te])),
                     id=1 - roc_auc_score(yte, f(P_trunc[te])))
            if DS: r["deepset"] = 1 - roc_auc_score(OOD_y[DS], f(OOD_trunc[DS]))
            rows.append(r)

res = pd.DataFrame(rows)
res = pd.concat([res, res[res.K == 0].assign(mode="benign_plus_attacks")], ignore_index=True)
for metric in ["itw_heldout", "id"] + (["deepset"] if DS else []):
    print(f"\n=== {metric}: error = 1 - AUROC, mean over {len(SEEDS_C)} seeds; columns = K extra itw prompts ===")
    print(res.pivot_table(index=["mode", "model", "layer"], columns="K", values=metric, aggfunc="mean").round(3).to_string())
print("\nIf itw_heldout error drops sharply with K (esp. benign_only) while id/deepset stay flat, the itw floor is benign-prompt coverage.")

In [ ]:
# 13b. Extract mean-pooled features for ALL layers (cached to Drive as float16). Only needed once.
@torch.no_grad()
def run_model_mean_all(model, tok, ids_list, bs=16):
    out_all = []
    for i in range(0, len(ids_list), bs):
        batch = ids_list[i:i + bs]; L = max(map(len, batch))
        inp = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for j, ids in enumerate(batch):
            inp[j, :len(ids)] = torch.tensor(ids); att[j, :len(ids)] = 1
        out = model(input_ids=inp.to(DEVICE), attention_mask=att.to(DEVICE), output_hidden_states=True)
        a = att.to(DEVICE).unsqueeze(-1).float()
        per_layer = [((h.float() * a).sum(1) / a.sum(1)) for h in out.hidden_states]   # each [B, d]
        out_all.append(torch.stack(per_layer, dim=1).cpu())                              # [B, n_layers+1, d]
    return torch.cat(out_all).numpy().astype(np.float16)

for name, hf in MODELS.items():
    todo = [("meanall_clean", P)] + [(f"meanall_{OOD_COND[k]}", o_.prompt.values) for k, o_ in OOD.items()]
    todo = [x for x in todo if not os.path.exists(fpath(name, x[0]))]
    if not todo: continue
    print(f"[{name}] all-layer mean-pool extracting: {[x[0] for x in todo]}")
    tok = AutoTokenizer.from_pretrained(hf); tok.padding_side = "right"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModel.from_pretrained(hf, torch_dtype=torch.float32).to(DEVICE).eval()
    for cond, texts in todo:
        np.savez(fpath(name, cond), X=run_model_mean_all(model, tok, encode(tok, texts)))
    del model; torch.cuda.empty_cache()
print("all-layer features cached.")

In [ ]:
# 13c. Layer sweep: pick the layer on IN-DISTRIBUTION validation only (grouped CV on train), then report OOD for that layer.
from sklearn.model_selection import GroupKFold

ALL = {}
for name in MODELS:
    ALL[name] = {"clean": np.load(fpath(name, "meanall_clean"))["X"]}
    for k in OOD: ALL[name][f"ood:{k}"] = np.load(fpath(name, f"meanall_{OOD_COND[k]}"))["X"]

def fast_probe(X, y, C=0.01):
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(C=C, max_iter=1000).fit(sc.transform(X), y)
    return lambda Z: clf.decision_function(sc.transform(Z))

gkf = GroupKFold(n_splits=5); groups_tr = df.group.values[tr]
rows = []
for name in MODELS:
    A = ALL[name]; nL = A["clean"].shape[1]
    for l in range(nL):
        Xl = A["clean"][:, l, :].astype(np.float32); Xtr_, Xte_ = Xl[tr], Xl[te]
        cv = []
        for a_, b_ in gkf.split(Xtr_, ytr, groups_tr):
            if len(set(ytr[b_])) < 2: continue
            cv.append(roc_auc_score(ytr[b_], fast_probe(Xtr_[a_], ytr[a_])(Xtr_[b_])))
        f = fast_probe(Xtr_, ytr)
        r = dict(model=name, layer=l, n_layers=nL - 1, cv_auc=np.mean(cv), id=roc_auc_score(yte, f(Xte_)))
        for k in OOD: r[k] = roc_auc_score(OOD_y[k], f(A[f"ood:{k}"][:, l, :].astype(np.float32)))
        rows.append(r)
prof = pd.DataFrame(rows)

# layer chosen by in-distribution CV only
sel = prof.loc[prof.groupby("model").cv_auc.idxmax()].set_index("model")
out = []
for name in MODELS:
    li = META[name]["layer_idx"]
    for tag, l in [("selected(ID-CV)", int(sel.loc[name, "layer"])), ("mid", li["mid"]), ("last", li["last"])]:
        r = prof[(prof.model == name) & (prof.layer == l)].iloc[0]
        out.append({"model": name, "which": tag, "layer": l, "cv_auc": r.cv_auc, "id": r.id, **{k: r[k] for k in OOD}})
print("Same fast probe (C=0.01) for every row. 'selected' uses ONLY in-distribution grouped-CV on train.")
print(pd.DataFrame(out).set_index(["model", "which"]).round(3).to_string())

# descriptive layer profile (not used for selection)
cols = ["cv_auc", "id"] + list(OOD)
fig, axes = plt.subplots(1, len(cols), figsize=(4.2 * len(cols), 3.6), squeeze=False)
for ax, c in zip(axes[0], cols):
    for name in MODELS:
        g = prof[prof.model == name]; ax.plot(g.layer / g.n_layers, g[c], "o-", ms=3, label=name)
        ax.scatter([sel.loc[name, "layer"] / sel.loc[name, "n_layers"]], [g[g.layer == sel.loc[name, "layer"]][c].iloc[0]],
                   marker="*", s=120, edgecolor="k", zorder=5)
    ax.set_xlabel("relative depth"); ax.set_title(f"AUROC: {c}")
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("layer_sweep.png", dpi=150, bbox_inches="tight"); plt.show()